<!-- codex_annotation: script_overview -->
# 局部限幅模板匹配测试

对 DAPI tile 进行模板匹配，并通过阈值和局部搜索思路辅助检查匹配质量，适合调试坐标生成参数。

注释说明：
- SCORE_THRESHOLD 用于标记低置信度匹配。
- 若匹配跑偏，可结合已知 tile 排列调整搜索范围。


In [10]:
import os
import cv2
import tifffile as tiff
import numpy as np

# ==========================
# 路径设置
# ==========================
# 更改为你需要的 H 样本或 A 样本的完整参考图
FULL_IMAGE = r"D:\01.analysis\11.test_result\MAX_C1-P4-rep2-A.tif" 
TILE_DIR = r"D:\01.analysis\11.test_result\DAPI"
OUTPUT_FILE = r"D:\01.analysis\11.test_result\DAPI\TileConfiguration.txt"

print("Loading full brain...")
full = tiff.imread(FULL_IMAGE)
if full.ndim > 2:    
    full = full[0]
full = cv2.normalize(full, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
H_full, W_full = full.shape
print("Full brain size:", full.shape)

tile_files = sorted([f for f in os.listdir(TILE_DIR) if f.endswith(".tif")])
results = []

# ==========================
# 参数配置（根据你的实际拍摄进行调整）
# ==========================
# 假设大体知道一列有多少个tile，或者重叠率（如 10%）
# 或者是通过已知的相邻估计，这里使用局部限幅防止跑偏
# 如果完全不知道大体位置，可以先提高匹配阈值
SCORE_THRESHOLD = 0.5  

for tile_name in tile_files:
    print(f"\nSearching: {tile_name}")
    tile_path = os.path.join(TILE_DIR, tile_name)
    tile = tiff.imread(tile_path)
    if tile.ndim > 2:        
        tile = tile[0]
    tile = cv2.normalize(tile, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    h_t, w_t = tile.shape

    # 全局匹配
    result = cv2.matchTemplate(full, tile, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)
    x, y = max_loc
    
    # 【预警机制】如果得分太低，说明可能匹配到了错误位置
    if max_val < SCORE_THRESHOLD:
        print(f" [WARNING] Low score ({max_val:.3f}) for {tile_name}. Position might be wrong.")
        # 这里可以加入逻辑：如果是连续扫描，可以基于上一个 tile 的坐标加上固定的位移量(offset)来强行赋予初始值
    
    print(f"Position = ({x},{y}) Score={max_val:.3f}")
    results.append((tile_name, x, y))

# ==========================
# 输出 Fiji 坐标文件
# ==========================
with open(OUTPUT_FILE, "w") as f:
    f.write("dim = 2\n\n")
    for tile_name, x, y in results:
        f.write(f"{tile_name}; ; ({x}.0, {y}.0)\n") # 加上 .0 符合 Fiji 浮点数习惯

print("\nSaved:", OUTPUT_FILE)

Loading full brain...
Full brain size: (11120, 17303)

Searching: tile_01.tif
 [WARNING] Low score (0.398) for tile_01.tif. Position might be wrong.
Position = (9477,1546) Score=0.398

Searching: tile_02.tif
 [WARNING] Low score (0.249) for tile_02.tif. Position might be wrong.
Position = (3721,13) Score=0.249

Searching: tile_04.tif


: 